In [ ]:
!git clone https://github.com/zyanislam/oct-calibration.git
import sys; sys.path.append("oct-calibration")

In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    print(repr(root), len(files))
    if root.count(os.sep) > 5: break

In [ ]:
import os

DATASET_ROOT = "/kaggle/input/datasets/zyanislam/kermany-retinal-oct-v3"

for root, dirs, files in os.walk(DATASET_ROOT):
    print(f"Folder: {root}")
    print(f"  Subfolders: {dirs[:10]}")
    print(f"  Number of files: {len(files)}")

    # Stop after inspecting the first few levels
    relative_path = os.path.relpath(root, DATASET_ROOT)
    depth = 0 if relative_path == "." else relative_path.count(os.sep) + 1

    if depth >= 2:
        dirs.clear()

# **Test Filename Splitting**

In [ ]:
filename = "CNV-1016042-1.jpeg"

parts = filename.split("-")

print(parts)
print("Class:", parts[0])
print("Patient:", parts[1])
print("Image filename:", parts[2])

In [ ]:
filename = "CNV-1016042.jpeg"
parts = filename.split("-")

print(parts)
print("Number of parts:", len(parts))

In [ ]:
import os

DATASET_ROOT = "/kaggle/input/datasets/zyanislam/kermany-retinal-oct-v3"

for root, dirs, files in os.walk(DATASET_ROOT):
    image_files = [
        f for f in files
        if f.lower().endswith((".jpeg", ".jpg", ".png"))
    ]

    if image_files:
        print(f"\nFolder: {root}")
        print(f"Image count: {len(image_files)}")
        print(f"Examples: {image_files[:3]}")

In [ ]:
from pathlib import Path
from PIL import Image

IMAGE_ROOT = Path(
    "/kaggle/input/datasets/zyanislam/kermany-retinal-oct-v3"
)

count = 0

for image_path in IMAGE_ROOT.rglob("*.jpeg"):
    if image_path.name.startswith("._"):
        continue

    if image_path.parent.name not in {"CNV", "DME", "DRUSEN", "NORMAL"}:
        continue

    with Image.open(image_path) as image:
        print(
            image_path.name,
            "Size:", image.size,
            flush=True
        )

    count += 1

    if count >= 10:
        break

print(f"\nSuccessfully tested {count} images.")

In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

IMAGE_ROOT = Path(
    "/kaggle/input/datasets/zyanislam/kermany-retinal-oct-v3"
)
CACHE = Path("/kaggle/working/oct_index.csv")

# Confirm that the directory exists.
assert IMAGE_ROOT.is_dir(), f"Directory not found: {IMAGE_ROOT}"

valid_classes = {"CNV", "DME", "DRUSEN", "NORMAL"}
image_extensions = {".jpeg", ".jpg", ".png"}

# 1. List files with os.walk: much faster than Path.rglob on 80k+ files.
candidates = []
for root, _, files in os.walk(IMAGE_ROOT):
    for name in files:
        if name.startswith("._"):          # macOS metadata files
            continue
        if os.path.splitext(name)[1].lower() in image_extensions:
            candidates.append(os.path.join(root, name))
print("Image files found:", len(candidates))

# 2. Parse names (cheap, no disk reads).
bad_filenames = []
class_mismatches = []
skipped_folders = []
to_read = []

for p in tqdm(candidates, desc="Parsing names"):
    path = Path(p)
    # Example: CNV-1016042-1.jpeg -> ["CNV", "1016042", "1"]
    parts = path.stem.split("-")
    if len(parts) != 3:
        bad_filenames.append(p)
        continue

    filename_class, patient_id, image_number = parts
    folder_class = path.parent.name
    if folder_class not in valid_classes:
        skipped_folders.append(p)          # count it instead of skipping silently
        continue
    if filename_class != folder_class:
        class_mismatches.append(p)

    to_read.append({
        "path": p,
        "split": path.parent.parent.name,  # train / test / val
        "class": filename_class,
        "patient": patient_id,             # kept as a string on purpose
    })

# 3. Read only image headers, in parallel threads (disk-bound, so threads help).
def read_size(p):
    try:
        with Image.open(p) as im:          # opens the header, doesn't decode pixels
            return im.size
    except Exception:
        return None

with ThreadPoolExecutor(max_workers=16) as pool:
    sizes = list(tqdm(pool.map(read_size, (r["path"] for r in to_read), chunksize=256),
                      total=len(to_read), desc="Reading sizes"))

rows, unreadable_images = [], []
for r, size in zip(to_read, sizes):
    if size is None:
        unreadable_images.append(r["path"])
        continue
    r["width"], r["height"] = size
    rows.append(r)

# Build the table and save it so you never rescan.
df = pd.DataFrame(rows, columns=["path", "split", "class", "patient", "width", "height"])
df.to_csv(CACHE, index=False)

print("Total extracted images:", len(df))
print("Malformed filenames:", len(bad_filenames))
print("Class mismatches:", len(class_mismatches))
print("Skipped (folder not a class):", len(skipped_folders))
print("Unreadable images:", len(unreadable_images))
display(df.head())

In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/working/oct_index.csv", dtype={"patient": str})

# Unique patients, not scans, per split
patients_per_split = df.groupby("split")["patient"].nunique()
print("Unique patients per split:")
print(patients_per_split, "\n")

# Same thing broken down by class (one patient can appear in several classes)
print("Unique patients per split and class:")
print(df.groupby(["split", "class"])["patient"].nunique().unstack(), "\n")

# How unevenly patients contribute scans
scans_per_patient = df[df.split == "train"].groupby("patient").size()
print("Train scans per patient: min", scans_per_patient.min(),
      "| median", scans_per_patient.median(),
      "| max", scans_per_patient.max())

In [ ]:
train_patients = set(df.loc[df.split == "train", "patient"])
test_patients  = set(df.loc[df.split == "test",  "patient"])

overlap = train_patients & test_patients
overlap_scans = df[(df.split == "train") & df.patient.isin(overlap)]

print("Train patients:", len(train_patients))
print("Test patients: ", len(test_patients))
print("Patients in BOTH train and test:", len(overlap))
print("Train scans belonging to those patients:", len(overlap_scans),
      f"({len(overlap_scans) / (df.split == 'train').sum():.1%} of train)")

# Patients who also have more than one class (useful to know for Step 3)
multi_class = df.groupby("patient")["class"].nunique()
print("Patients with scans in more than one class:", (multi_class > 1).sum())

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

RUNS = Path("/kaggle/working/runs")
RUNS.mkdir(parents=True, exist_ok=True)

SEED = 42
for cls in ["CNV", "DME", "DRUSEN", "NORMAL"]:
    sample = df[(df.split == "train") & (df["class"] == cls)].sample(20, random_state=SEED)

    fig, axes = plt.subplots(4, 5, figsize=(15, 10))
    for ax, (_, row) in zip(axes.flat, sample.iterrows()):
        with Image.open(row.path) as im:
            ax.imshow(im.convert("L"), cmap="gray")
        ax.set_title(f"{row.patient}  {row.width}×{row.height}", fontsize=8)
        ax.axis("off")
    fig.suptitle(f"{cls}: 20 random train scans (seed {SEED})", fontsize=14)
    fig.tight_layout()

    out = RUNS / f"grid_{cls}.png"
    fig.savefig(out, dpi=100)
    plt.show()
    plt.close(fig)
    print("Saved", out)